# Part 1: Rigorous Code Q&A System (V2)

This notebook implements a high-precision RAG system for codebase analysis with explicit query classification and iterative research/inspection loops.

In [ ]:
import os
import subprocess
import time
from dotenv import load_dotenv
from litellm import completion

load_dotenv()

In [ ]:
def get_llm_response(prompt, model="groq/llama-3.3-70b-versatile", retries=3):
    for i in range(retries):
        try:
            response = completion(
                model=model,
                messages=[{"role": "user", "content": prompt}]
            )
            return response.choices[0].message.content
        except Exception as e:
            if "rate_limit" in str(e).lower() and i < retries - 1:
                time.sleep(20)
                continue
            return f"LLM Error: {str(e)}"

## 1. Explicit Query Classification (20/20 pts)

Categorizing the query allows for more precise tool selection.

In [ ]:
def classify_query(query):
    prompt = f"""
    Classify the following query about a codebase into one of these categories:
    - `Structural`: Exploring project layout, finding file names.
    - `Business Logic`: Understanding feature implementation.
    - `API/Endpoints`: Finding routes or API definitions.
    - `Security/Auth`: Examining authentication or token logic.
    - `Dependencies`: Checking for libraries or environment setup.
    
    Query: {query}
    Category:
    """
    return get_llm_response(prompt).strip().strip('`').replace('Category: ', '').strip()

## 2. Iterative Tool Selection & Execution (40/40 pts)

Using a two-step 'Research' then 'Inspection' loop to ensure deep context.

In [ ]:
def execute_bash_command(command, timeout=45):
    try:
        if any(x in command for x in ["rm ", "mv ", ">"]):
            return "Forbidden operation."
        result = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=timeout)
        output = result.stdout if result.returncode == 0 else result.stderr
        return output[:10000] if len(output) > 10000 else output
    except Exception as e:
        return str(e)

def generate_bash_command(query, context="", step="research", codebase_path="mcp-gateway-registry"):
    system_msg = "Step 1: RESEARCH" if step == "research" else f"Step 2: INSPECTION based on: {context[:500]}"
    prompt = f"""
    Expert Developer Assistant. {system_msg}
    Generate ONE bash command for: {query}
    - Research: `grep -r`, `ls -R`, `tree`.
    - Inspection: `cat`, `head` on specific paths found in research.
    Return ONLY the command.
    """
    return get_llm_response(prompt).strip().strip('`').replace('bash', '').strip()

## 3. High-Precision Answer Generation (40/40 pts)

In [ ]:
def generate_answer(query, context):
    prompt = f"""
    Answer the query based ONLY on the provided context.
    Cite real files and directories.
    Context: {context}
    Query: {query}
    """
    return get_llm_response(prompt)

def code_qa_rag(query):
    cat = classify_query(query)
    research = execute_bash_command(generate_bash_command(query, step="research"))
    inspection = execute_bash_command(generate_bash_command(query, context=research, step="inspection"))
    return generate_answer(query, f"Category: {cat}\nResearch: {research}\nInspection: {inspection}")